In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch

print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
print("current device:", torch.cuda.current_device() if torch.cuda.is_available() else None)
print("device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

cuda available: True
device count: 1
current device: 0
device name: Tesla T4


In [ ]:

import os, math, cv2, numpy as np, torch, torchvision
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# ----------------------------
# Config
# ----------------------------
ROOT = "/content/drive/MyDrive/annotated_images"
IMG_SIZE = 224           # we'll still crop/resize to 224x224 before feeding the detector
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# for reproducible train/val split
torch.manual_seed(42)
np.random.seed(42)

# ----------------------------
# Helpers
# ----------------------------
def parse_annotation(txt_path, img_w, img_h):
    """
    Accepts two common formats per line:
      1) x y r v  (pixels; v in {0,1})
      2) xmin ymin xmax ymax [v]  (pixels; optional v)
    Returns list of [x, y, r, v] in ORIGINAL image coords.
    """
    balls = []
    if not os.path.exists(txt_path):
        return balls
    with open(txt_path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            parts = line.replace(",", " ").split()
            nums = []
            for t in parts:
                try:
                    nums.append(float(t))
                except ValueError:
                    pass
            if len(nums) >= 4:
                # If it looks like circle: x y r [v]
                # Heuristic: r shouldn't exceed 40% of min(img_w,img_h)
                x, y, a, b = nums[0], nums[1], nums[2], (nums[3] if len(nums) > 3 else 0.0)
                if a > 0 and a < 0.5 * min(img_w, img_h) and (len(nums) in (3, 4, 5)):
                    r = a
                    v = int(round(b)) if len(nums) >= 4 else 0
                    balls.append([x, y, r, v])
                else:
                    # Treat as box: xmin ymin xmax ymax [v]
                    xmin, ymin, xmax, ymax = nums[0], nums[1], nums[2], nums[3]
                    cx = (xmin + xmax) / 2.0
                    cy = (ymin + ymax) / 2.0
                    r = 0.5 * min(max(0.0, xmax - xmin), max(0.0, ymax - ymin))
                    v = int(round(nums[4])) if len(nums) >= 5 else 0
                    balls.append([cx, cy, r, v])
    return balls

def center_square_crop_and_resize(img, balls, out_size=224):
    """
    Center-square crop, then resize to out_size x out_size.
    Adjust circle centers/radii accordingly. Keep balls whose centers fall into the crop.
    Returns resized_image, resized_balls, (off_x, off_y, siz, scale)
    """
    h, w = img.shape[:2]
    siz = min(h, w)
    off_x = (w - siz) // 2
    off_y = (h - siz) // 2
    scale = out_size / float(siz)

    keep = []
    for (x, y, r, v) in balls:
        if x < off_x or x >= off_x + siz or y < off_y or y >= off_y + siz:
            continue  # center outside crop
        x2 = (x - off_x) * scale
        y2 = (y - off_y) * scale
        r2 = r * scale
        keep.append([x2, y2, r2, v])

    crop = img[off_y:off_y+siz, off_x:off_x+siz]
    resized = cv2.resize(crop, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    return resized, keep, (off_x, off_y, siz, scale)

# ----------------------------
# Dataset (adapted for Faster R-CNN)
# ----------------------------
class BallDataset(Dataset):
    def __init__(self, root=ROOT, img_size=IMG_SIZE):
        self.root = root
        self.img_size = img_size
        self.images = sorted(
            [f for f in os.listdir(root) if f.lower().endswith(".png")],
            key=lambda n: int(os.path.splitext(n)[0]) if os.path.splitext(n)[0].isdigit() else n
        )
        # IMPORTANT: For torchvision detection models, only ToTensor();
        # they will handle normalization internally.
        self.tf = torchvision.transforms.ToTensor()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        fname = self.images[idx]
        img_path = os.path.join(self.root, fname)
        txt_path = os.path.join(self.root, os.path.splitext(fname)[0] + ".txt")

        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(img_path)
        h, w = img_bgr.shape[:2]
        balls = parse_annotation(txt_path, w, h)

        # crop and resize image + adjust ball positions
        img_bgr, balls_resized, _ = center_square_crop_and_resize(img_bgr, balls, out_size=self.img_size)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        x = self.tf(img_rgb)  # (3,H,W), float32 in [0,1]

        # build Faster R-CNN targets: boxes + labels
        boxes = []
        labels = []
        for (cx, cy, r, v) in balls_resized:
            # convert circle to bounding box
            xmin = max(cx - r, 0.0)
            ymin = max(cy - r, 0.0)
            xmax = min(cx + r, self.img_size - 1.0)
            ymax = min(cy + r, self.img_size - 1.0)
            if xmax <= xmin or ymax <= ymin:
                continue
            boxes.append([xmin, ymin, xmax, ymax])
            # v = 0 solid, v = 1 striped -> labels 1 and 2 (background is 0)
            label = 1 if v == 0 else 2
            labels.append(label)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.zeros((0,), dtype=torch.float32),
            "iscrowd": torch.zeros((boxes.shape[0],), dtype=torch.int64),
        }

        return x, target

# custom collate fn for detection models
def collate_fn(batch):
    images, targets = list(zip(*batch))
    return list(images), list(targets)

# ----------------------------
# Dataset & Train/Val split
# ----------------------------
full_dataset = BallDataset(ROOT, IMG_SIZE)

# 80% train, 20% validation (change ratio if you like)
val_ratio = 0.2
n_total = len(full_dataset)
n_val = int(round(n_total * val_ratio))
n_train = n_total - n_val

train_dataset, val_dataset = random_split(full_dataset, [n_train, n_val])

print(f"Total images: {n_total}  |  Train: {n_train}  |  Val: {n_val}")

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

# ----------------------------
# Faster R-CNN model
# ----------------------------
# load COCO-pretrained Faster R-CNN
model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
# we want 3 classes: background(0), solid ball(1), striped ball(2)
num_classes = 3
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def train_epoch():
    model.train()
    loss_sum = 0.0
    n_batches = 0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        loss_sum += losses.item()
        n_batches += 1

    return loss_sum / max(1, n_batches)

def eval_epoch():
    """
    Compute validation loss (no gradient). For torchvision detection
    we need model in train mode to return losses, but we wrap in no_grad()
    so weights are not updated. With FrozenBatchNorm in detection models,
    this is safe.
    """
    model.train()  # to make it return losses given targets
    loss_sum = 0.0
    n_batches = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            loss_sum += losses.item()
            n_batches += 1

    avg_loss = loss_sum / max(1, n_batches)
    model.eval()
    return avg_loss

# quick sanity pass
model.eval()
with torch.no_grad():
    imgs, targs = next(iter(train_loader))
    preds = model([imgs[0].to(device)])
    print("Sample prediction keys:", preds[0].keys())  # boxes, labels, scores

# ----------------------------
# Training loop with best-model tracking
# ----------------------------
EPOCHS = 100
BEST_FULL_PATH = "balldetector_frcnn_best_full.pth"  # full precision
FINAL_HALF_PATH = "balldetector_frcnn.pth"           # half precision for deployment

best_val_loss = float("inf")
best_epoch = -1

for epoch in range(EPOCHS):
    train_loss = train_epoch()
    val_loss = eval_epoch()

    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")
    else:
        print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    # Save best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), BEST_FULL_PATH)
        print(f"  --> New best model at epoch {epoch} (val_loss={val_loss:.4f})")

print(f"Training done. Best epoch: {best_epoch} with val_loss={best_val_loss:.4f}")

# ----------------------------
# Load best weights and save in half precision
# ----------------------------
print("Loading best model and saving half-precision weights...")

best_state = torch.load(BEST_FULL_PATH, map_location="cpu")
model.load_state_dict(best_state)

model.eval()
model = model.cpu()
model = model.half()  # convert weights to float16 for smaller file
torch.save(model.state_dict(), FINAL_HALF_PATH)
print("Saved half-precision best model to:", FINAL_HALF_PATH)

Total images: 40  |  Train: 32  |  Val: 8
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 192MB/s]


Sample prediction keys: dict_keys(['boxes', 'labels', 'scores'])
Epoch   0 | train_loss=1.1944 | val_loss=0.7744
  --> New best model at epoch 0 (val_loss=0.7744)
Epoch   1 | train_loss=0.8247 | val_loss=0.6477
  --> New best model at epoch 1 (val_loss=0.6477)
Epoch   2 | train_loss=0.6944 | val_loss=0.6068
  --> New best model at epoch 2 (val_loss=0.6068)
Epoch   3 | train_loss=0.5978 | val_loss=0.6087
Epoch   4 | train_loss=0.5212 | val_loss=0.5479
  --> New best model at epoch 4 (val_loss=0.5479)
Epoch   5 | train_loss=0.4516 | val_loss=0.5024
  --> New best model at epoch 5 (val_loss=0.5024)
Epoch   6 | train_loss=0.4082 | val_loss=0.5198
Epoch   7 | train_loss=0.3670 | val_loss=0.5212
Epoch   8 | train_loss=0.3465 | val_loss=0.5418
Epoch   9 | train_loss=0.3609 | val_loss=0.5652
Epoch  10 | train_loss=0.3315 | val_loss=0.4813
  --> New best model at epoch 10 (val_loss=0.4813)
Epoch  11 | train_loss=0.2868 | val_loss=0.5606
Epoch  12 | train_loss=0.2807 | val_loss=0.5069
Epoch  13 

In [ ]:
from google.colab import files
files.download("balldetector_frcnn.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, math, cv2, numpy as np, torch, torchvision
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms

# ----------------------------
# Config
# ----------------------------
ROOT = "/content/drive/MyDrive/BallProject/annotated_images"
IMG_SIZE = 224           # crop/resize size for detector
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
np.random.seed(42)

# ============================================================
#  COMMON HELPERS
# ============================================================
def parse_annotation(txt_path, img_w, img_h):
    """
    Accepts two common formats per line:
      1) x y r v  (pixels; v in {0,1})
      2) xmin ymin xmax ymax [v]  (pixels; optional v)
    Returns list of [x, y, r, v] in ORIGINAL image coords.
    """
    balls = []
    if not os.path.exists(txt_path):
        return balls
    with open(txt_path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            parts = line.replace(",", " ").split()
            nums = []
            for t in parts:
                try:
                    nums.append(float(t))
                except ValueError:
                    pass
            if len(nums) >= 4:
                # If it looks like circle: x y r [v]
                x, y, a, b = nums[0], nums[1], nums[2], (nums[3] if len(nums) > 3 else 0.0)
                if a > 0 and a < 0.5 * min(img_w, img_h) and (len(nums) in (3, 4, 5)):
                    r = a
                    v = int(round(b)) if len(nums) >= 4 else 0
                    balls.append([x, y, r, v])
                else:
                    # Treat as box: xmin ymin xmax ymax [v]
                    xmin, ymin, xmax, ymax = nums[0], nums[1], nums[2], nums[3]
                    cx = (xmin + xmax) / 2.0
                    cy = (ymin + ymax) / 2.0
                    r = 0.5 * min(max(0.0, xmax - xmin), max(0.0, ymax - ymin))
                    v = int(round(nums[4])) if len(nums) >= 5 else 0
                    balls.append([cx, cy, r, v])
    return balls

def center_square_crop_and_resize(img, balls, out_size=224):
    """
    Center-square crop, then resize to out_size x out_size.
    Adjust circle centers/radii accordingly. Keep balls whose centers fall into the crop.
    Returns resized_image, resized_balls, (off_x, off_y, siz, scale)
    """
    h, w = img.shape[:2]
    siz = min(h, w)
    off_x = (w - siz) // 2
    off_y = (h - siz) // 2
    scale = out_size / float(siz)

    keep = []
    for (x, y, r, v) in balls:
        if x < off_x or x >= off_x + siz or y < off_y or y >= off_y + siz:
            continue  # center outside crop
        x2 = (x - off_x) * scale
        y2 = (y - off_y) * scale
        r2 = r * scale
        keep.append([x2, y2, r2, v])

    crop = img[off_y:off_y+siz, off_x:off_x+siz]
    resized = cv2.resize(crop, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    return resized, keep, (off_x, off_y, siz, scale)

# ============================================================
#  DETECTOR DATASET (Faster R-CNN)
# ============================================================
class BallDataset(Dataset):
    def __init__(self, root=ROOT, img_size=IMG_SIZE):
        self.root = root
        self.img_size = img_size
        self.images = sorted(
            [f for f in os.listdir(root) if f.lower().endswith(".png")],
            key=lambda n: int(os.path.splitext(n)[0]) if os.path.splitext(n)[0].isdigit() else n
        )
        self.tf = torchvision.transforms.ToTensor()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        fname = self.images[idx]
        img_path = os.path.join(self.root, fname)
        txt_path = os.path.join(self.root, os.path.splitext(fname)[0] + ".txt")

        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(img_path)
        h, w = img_bgr.shape[:2]
        balls = parse_annotation(txt_path, w, h)

        # crop and resize image + adjust ball positions
        img_bgr, balls_resized, _ = center_square_crop_and_resize(img_bgr, balls, out_size=self.img_size)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        x = self.tf(img_rgb)  # (3,H,W), float32 in [0,1]

        # build Faster R-CNN targets: boxes + labels
        boxes = []
        labels = []
        for (cx, cy, r, v) in balls_resized:
            xmin = max(cx - r, 0.0)
            ymin = max(cy - r, 0.0)
            xmax = min(cx + r, self.img_size - 1.0)
            ymax = min(cy + r, self.img_size - 1.0)
            if xmax <= xmin or ymax <= ymin:
                continue
            boxes.append([xmin, ymin, xmax, ymax])
            # v = 0 solid, v = 1 striped -> labels 1 and 2 (background is 0)
            label = 1 if v == 0 else 2
            labels.append(label)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.zeros((0,), dtype=torch.float32),
            "iscrowd": torch.zeros((boxes.shape[0],), dtype=torch.int64),
        }

        return x, target

def collate_fn(batch):
    images, targets = list(zip(*batch))
    return list(images), list(targets)

# build detector datasets
full_dataset = BallDataset(ROOT, IMG_SIZE)
val_ratio = 0.2
n_total = len(full_dataset)
n_val = int(round(n_total * val_ratio))
n_train = n_total - n_val

train_dataset, val_dataset = random_split(full_dataset, [n_train, n_val])

print(f"[Detector] Total images: {n_total} | Train: {n_train} | Val: {n_val}")

train_loader = DataLoader(
    train_dataset, batch_size=4, shuffle=True,
    num_workers=0, collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False,
    num_workers=0, collate_fn=collate_fn
)

# ============================================================
#  DETECTOR MODEL & TRAINING
# ============================================================
model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
num_classes = 3  # background(0), solid(1), striped(2)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def train_epoch_det():
    model.train()
    loss_sum = 0.0
    n_batches = 0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        loss_sum += losses.item()
        n_batches += 1

    return loss_sum / max(1, n_batches)

def eval_epoch_det():
    # detection models return losses only in "train" mode with targets
    model.train()
    loss_sum = 0.0
    n_batches = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            loss_sum += losses.item()
            n_batches += 1

    avg_loss = loss_sum / max(1, n_batches)
    model.eval()
    return avg_loss

# sanity check
model.eval()
with torch.no_grad():
    imgs, targs = next(iter(train_loader))
    preds = model([imgs[0].to(device)])
    print("[Detector] Sample prediction keys:", preds[0].keys())

EPOCHS_DET = 100
DETECTOR_BEST_FULL = "balldetector_frcnn_best_full.pth"
DETECTOR_HALF = "balldetector_frcnn.pth"

best_val_loss = float("inf")
best_epoch_det = -1

for epoch in range(EPOCHS_DET):
    train_loss = train_epoch_det()
    val_loss = eval_epoch_det()
    print(f"[Detector] Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch_det = epoch
        torch.save(model.state_dict(), DETECTOR_BEST_FULL)
        print(f"  --> New best detector at epoch {epoch} (val_loss={val_loss:.4f})")

print(f"[Detector] Training done. Best epoch: {best_epoch_det} with val_loss={best_val_loss:.4f}")

# save best detector in half precision
best_state = torch.load(DETECTOR_BEST_FULL, map_location="cpu")
model.load_state_dict(best_state)
model.eval()
model = model.cpu().half()
torch.save(model.state_dict(), DETECTOR_HALF)
print(f"[Detector] Saved half-precision best detector to: {DETECTOR_HALF}")

# ============================================================
#  CLASSIFIER: BUILD BALL PATCH SAMPLES
# ============================================================
def collect_ball_samples(root):
    """
    Returns a list of (img_path, x, y, r, v) for each annotated ball.
    v in {0,1} (solid/striped).
    """
    samples = []
    image_files = sorted(
        [f for f in os.listdir(root) if f.lower().endswith(".png")],
        key=lambda n: int(os.path.splitext(n)[0]) if os.path.splitext(n)[0].isdigit() else n
    )
    for fname in image_files:
        img_path = os.path.join(root, fname)
        txt_path = os.path.join(root, os.path.splitext(fname)[0] + ".txt")
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img is None:
            continue
        h, w = img.shape[:2]
        balls = parse_annotation(txt_path, w, h)
        for (x, y, r, v) in balls:
            if r <= 0:
                continue
            if v not in (0, 1):
                continue
            samples.append((img_path, x, y, r, v))
    return samples

samples = collect_ball_samples(ROOT)
print(f"[Classifier] Total ball samples: {len(samples)}")
if len(samples) == 0:
    print("[Classifier] No samples found, skipping classifier training.")
    exit(0)

# split indices into train / val
n_total_cls = len(samples)
val_ratio_cls = 0.2
n_val_cls = int(round(n_total_cls * val_ratio_cls))
n_train_cls = n_total_cls - n_val_cls

perm = torch.randperm(n_total_cls).tolist()
train_indices = perm[:n_train_cls]
val_indices = perm[n_train_cls:]

print(f"[Classifier] Samples: total={n_total_cls} | train={n_train_cls} | val={n_val_cls}")

# ============================================================
#  CLASSIFIER DATASET WITH AUGMENTATIONS
# ============================================================
class BallPatchDataset(Dataset):
    def __init__(self, samples, indices, patch_size=224, scale=1.3, transform=None):
        self.samples = samples
        self.indices = indices
        self.patch_size = patch_size
        self.scale = scale
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        sample_idx = self.indices[idx]
        img_path, x, y, r, v = self.samples[sample_idx]
        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(img_path)
        h, w = img_bgr.shape[:2]

        half = r * self.scale
        x1 = int(max(0, x - half))
        y1 = int(max(0, y - half))
        x2 = int(min(w, x + half))
        y2 = int(min(h, y + half))

        if x2 <= x1 or y2 <= y1:
            x1, y1 = max(0, int(x) - 5), max(0, int(y) - 5)
            x2, y2 = min(w, int(x) + 5), min(h, int(y) + 5)

        patch = img_bgr[y1:y2, x1:x2]
        patch = cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)
        patch = cv2.resize(patch, (self.patch_size, self.patch_size), interpolation=cv2.INTER_LINEAR)

        if self.transform is not None:
            patch = self.transform(patch)
        else:
            # default: to tensor + normalize
            patch = transforms.ToTensor()(patch)
            patch = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)(patch)

        label = torch.tensor(v, dtype=torch.long)  # 0 solid, 1 striped
        return patch, label

# transforms
train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_cls_dataset = BallPatchDataset(samples, train_indices, patch_size=224, scale=1.3, transform=train_tf)
val_cls_dataset = BallPatchDataset(samples, val_indices, patch_size=224, scale=1.3, transform=val_tf)

print(f"[Classifier] Train patches: {len(train_cls_dataset)} | Val patches: {len(val_cls_dataset)}")

train_cls_loader = DataLoader(train_cls_dataset, batch_size=64, shuffle=True, num_workers=0)
val_cls_loader = DataLoader(val_cls_dataset, batch_size=64, shuffle=False, num_workers=0)

# ============================================================
#  CLASSIFIER MODEL & TRAINING
# ============================================================
cls_model = resnet18(weights=ResNet18_Weights.DEFAULT)
in_features_cls = cls_model.fc.in_features
cls_model.fc = torch.nn.Linear(in_features_cls, 2)  # solid(0), striped(1)
cls_model = cls_model.to(device)

# class weights based on train set
counts = [0, 0]
for idx in train_indices:
    _, _, _, _, v = samples[idx]
    counts[int(v)] += 1
total_cls = sum(counts)
w0 = total_cls / (2.0 * max(1, counts[0]))
w1 = total_cls / (2.0 * max(1, counts[1]))
class_weights = torch.tensor([w0, w1], dtype=torch.float32).to(device)
print(f"[Classifier] Class counts: solid={counts[0]} striped={counts[1]} | weights={w0:.2f},{w1:.2f}")

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer_cls = torch.optim.Adam(cls_model.parameters(), lr=1e-4)

def train_cls_epoch():
    cls_model.train()
    loss_sum = 0.0
    correct = 0
    total = 0
    for imgs, labels in train_cls_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        logits = cls_model(imgs)
        loss = criterion(logits, labels)

        optimizer_cls.zero_grad()
        loss.backward()
        optimizer_cls.step()

        loss_sum += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return loss_sum / max(1, total), correct / max(1, total)

def eval_cls_epoch():
    cls_model.eval()
    loss_sum = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in val_cls_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            logits = cls_model(imgs)
            loss = criterion(logits, labels)
            loss_sum += loss.item() * imgs.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
    return loss_sum / max(1, total), correct / max(1, total)

CLS_BEST_FULL = "balltype_resnet18_best_full.pth"
CLS_HALF = "balltype_resnet18.pth"
best_val_acc = 0.0
best_epoch_cls = -1
EPOCHS_CLS = 40  # more epochs than before

for epoch in range(EPOCHS_CLS):
    train_loss_c, train_acc_c = train_cls_epoch()
    val_loss_c, val_acc_c = eval_cls_epoch()
    print(f"[Classifier] Epoch {epoch:3d} | train_loss={train_loss_c:.4f} acc={train_acc_c*100:.1f}% | "
          f"val_loss={val_loss_c:.4f} acc={val_acc_c*100:.1f}%")

    if val_acc_c > best_val_acc:
        best_val_acc = val_acc_c
        best_epoch_cls = epoch
        torch.save(cls_model.state_dict(), CLS_BEST_FULL)
        print(f"  --> New best classifier at epoch {epoch} (val_acc={val_acc_c*100:.1f}%)")

print(f"[Classifier] Training done. Best epoch: {best_epoch_cls} with val_acc={best_val_acc*100:.1f}%")

best_cls_state = torch.load(CLS_BEST_FULL, map_location="cpu")
cls_model.load_state_dict(best_cls_state)
cls_model.eval()
cls_model = cls_model.cpu().half()
torch.save(cls_model.state_dict(), CLS_HALF)
print(f"[Classifier] Saved half-precision best classifier to: {CLS_HALF}")

[Detector] Total images: 40 | Train: 32 | Val: 8
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 202MB/s]


[Detector] Sample prediction keys: dict_keys(['boxes', 'labels', 'scores'])
[Detector] Epoch   0 | train_loss=1.1971 | val_loss=0.7431
  --> New best detector at epoch 0 (val_loss=0.7431)
[Detector] Epoch   1 | train_loss=0.8112 | val_loss=0.6302
  --> New best detector at epoch 1 (val_loss=0.6302)
[Detector] Epoch   2 | train_loss=0.6886 | val_loss=0.6003
  --> New best detector at epoch 2 (val_loss=0.6003)
[Detector] Epoch   3 | train_loss=0.5856 | val_loss=0.5847
  --> New best detector at epoch 3 (val_loss=0.5847)
[Detector] Epoch   4 | train_loss=0.5121 | val_loss=0.5605
  --> New best detector at epoch 4 (val_loss=0.5605)
[Detector] Epoch   5 | train_loss=0.4523 | val_loss=0.5320
  --> New best detector at epoch 5 (val_loss=0.5320)
[Detector] Epoch   6 | train_loss=0.3947 | val_loss=0.5342
[Detector] Epoch   7 | train_loss=0.3698 | val_loss=0.5051
  --> New best detector at epoch 7 (val_loss=0.5051)
[Detector] Epoch   8 | train_loss=0.3459 | val_loss=0.5156
[Detector] Epoch   9 |

100%|██████████| 44.7M/44.7M [00:00<00:00, 166MB/s]


[Classifier] Class counts: solid=222 striped=151 | weights=0.84,1.24
[Classifier] Epoch   0 | train_loss=0.7130 acc=51.5% | val_loss=0.5850 acc=69.9%
  --> New best classifier at epoch 0 (val_acc=69.9%)
[Classifier] Epoch   1 | train_loss=0.4367 acc=80.2% | val_loss=0.5543 acc=76.3%
  --> New best classifier at epoch 1 (val_acc=76.3%)
[Classifier] Epoch   2 | train_loss=0.2967 acc=89.0% | val_loss=0.5156 acc=77.4%
  --> New best classifier at epoch 2 (val_acc=77.4%)
[Classifier] Epoch   3 | train_loss=0.2079 acc=92.5% | val_loss=0.5520 acc=77.4%
[Classifier] Epoch   4 | train_loss=0.1698 acc=94.4% | val_loss=0.4887 acc=80.6%
  --> New best classifier at epoch 4 (val_acc=80.6%)
[Classifier] Epoch   5 | train_loss=0.1195 acc=94.9% | val_loss=0.3953 acc=83.9%
  --> New best classifier at epoch 5 (val_acc=83.9%)
[Classifier] Epoch   6 | train_loss=0.1116 acc=95.4% | val_loss=0.4266 acc=81.7%
[Classifier] Epoch   7 | train_loss=0.0848 acc=96.5% | val_loss=0.4668 acc=81.7%
[Classifier] Epoch

In [ ]:
from google.colab import files
files.download("balldetector_frcnn.pth")
files.download("balltype_resnet18.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>